# 🧱 Lab: Clone-and-Run Check

**Module 4: Escape the Notebook: Environment, Repo, Git** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **List** what makes a repo clone-and-run versus a repo that wastes a stranger's day
2. **Clone** into a new folder and rebuild the environment from pinned dependencies
3. **Prove** a tagged, known-good commit runs from a fresh clone

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Reproducibility | Someone else's machine reproduces your result from documented, pinned steps, starting from a fresh repo clone with no hidden setup. | This lab |
| Virtual environment | A virtual environment is an isolated workspace that keeps a project’s Python packages and settings separate, so installs and upgrades do not break other projects. | mini-env-setup |
| Dependency Pinning | Lock your project’s libraries to specific versions so installs are repeatable, bugs are reproducible, and deployments behave the same everywhere. | mini-pin-dependencies |
| Version control basics | Init a repo, ignore generated files, commit small changes, and tag v0 to v5 as known good checkpoints you can reset to safely. | mini-init-git-repo |
| Secrets Hygiene | Secrets hygiene is the habit of keeping passwords, API keys, and tokens out of code and logs, sharing them safely, and rotating them regularly so one mistake doesn’t become a breach. | mini-init-git-repo |

## Prerequisites

- **mini-pin-dependencies**: Pin and regenerate
- **mini-restructure-project**: src / tests / data / docs layout
- **mini-init-git-repo**: init, gitignore, commit


---
## Two repos, one checklist

Before you clone: a reproducible repo has pinned deps, a documented run command, and no secrets in git. A waste-your-day repo is missing one of those. The wall below is the test.


## What We're Doing

This lab is the **final checkpoint** for the module. Everything you learned — environments, pinning, project structure, Git, secrets hygiene — comes together in one test:

> **Can a stranger clone your repo and run the project without asking you a single question?**

We'll build a small but real project, set it up correctly, then simulate the "stranger's experience" by cloning it into a separate directory and running it from scratch. If it works, your repo is reproducible. If it doesn't, we'll diagnose why.

## Step 1 — Build the Reproducibility Checklist

Before we create anything, let's consolidate what we've learned into a concrete checklist. A repo is reproducible if — and only if — every item passes.

> 🧩 **Reproducibility.** Watch for how each checklist row removes one piece of hidden setup that a stranger's fresh clone would otherwise trip over.

In [4]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


checklist = """
### The Clone-and-Run Checklist

| # | Check | Why It Matters | Concept |
|---|-------|---------------|---------|
| 1 | `requirements.txt` exists with pinned versions (`==`) | Without it, `pip install` grabs random latest versions | Dependency Pinning |
| 2 | `.gitignore` includes `.env`, `.venv/`, `__pycache__/` | Keeps secrets and junk out of the repo | Secrets Hygiene |
| 3 | `.env.example` exists with placeholder keys | Tells the newcomer which env vars are needed | Secrets Hygiene |
| 4 | No secrets in any committed file | API keys in history = leaked keys | Secrets Hygiene |
| 5 | `src/` has the runnable entry point | The newcomer knows what to run | Project Structure |
| 6 | `README.md` has setup + run instructions | The newcomer doesn't have to guess | Reproducibility |
| 7 | `.venv/` is NOT committed | 100s of MB of binaries; recreated from `requirements.txt` | Virtual Environments |
| 8 | The project actually runs after the setup steps | The ultimate test | All of the above |
"""

md(checklist)


### The Clone-and-Run Checklist

| # | Check | Why It Matters | Concept |
|---|-------|---------------|---------|
| 1 | `requirements.txt` exists with pinned versions (`==`) | Without it, `pip install` grabs random latest versions | Dependency Pinning |
| 2 | `.gitignore` includes `.env`, `.venv/`, `__pycache__/` | Keeps secrets and junk out of the repo | Secrets Hygiene |
| 3 | `.env.example` exists with placeholder keys | Tells the newcomer which env vars are needed | Secrets Hygiene |
| 4 | No secrets in any committed file | API keys in history = leaked keys | Secrets Hygiene |
| 5 | `src/` has the runnable entry point | The newcomer knows what to run | Project Structure |
| 6 | `README.md` has setup + run instructions | The newcomer doesn't have to guess | Reproducibility |
| 7 | `.venv/` is NOT committed | 100s of MB of binaries; recreated from `requirements.txt` | Virtual Environments |
| 8 | The project actually runs after the setup steps | The ultimate test | All of the above |


## Step 2 — Create the Project

We'll build a small CLI tool that reads a message from an environment variable and prints a greeting. It's trivial on purpose — the point is the **infrastructure around it**, not the code itself.

Run every command below **in your terminal**, in order.

### 2a — Create the folder structure

```bash
mkdir clone-test-project
cd clone-test-project

mkdir src
mkdir tests
mkdir data
mkdir docs
```

### 2b — Create the entry point

Create the file `src/app.py` with this content:

```python
"""A tiny app that proves the project is reproducible."""
import os
from dotenv import load_dotenv
from rich.console import Console

load_dotenv()

def main():
    console = Console()
    project_name = os.getenv("PROJECT_NAME", "Unnamed Project")
    console.print(f"[bold green]✅ Clone-and-run succeeded![/bold green]")
    console.print(f"   Project: {project_name}")
    console.print(f"   Python:  {os.sys.executable}")
    console.print(f"   dotenv loaded, rich installed — dependencies work.")

if __name__ == "__main__":
    main()
```

This app intentionally uses two external packages (`python-dotenv` and `rich`) so we have real dependencies to pin.

### 2c — Create the virtual environment and install dependencies

```bash
python -m venv .venv

# Activate:
# macOS/Linux:
source .venv/bin/activate
# Windows:
# .venv\Scripts\activate

pip install python-dotenv rich
```

### 2d — Pin the dependencies

```bash
pip freeze > requirements.txt
```

Verify the file has `==` pinned versions:

```bash
cat requirements.txt
```

### 2e — Create `.env` and `.env.example`

Create `.env` (your real config — will be ignored by Git):

```
PROJECT_NAME=Clone Test Project
```

Create `.env.example` (committed to repo — tells newcomers what's needed):

```
# Copy this file to .env and fill in your values
PROJECT_NAME=Your Project Name Here
```

### 2f — Create `.gitignore`

Create `.gitignore` with:

```
.venv/
.env
__pycache__/
*.pyc
.vscode/
.idea/
.DS_Store
Thumbs.db
```

### 2g — Create `README.md`

Create `README.md` with:

```markdown
# Clone Test Project

A minimal project that proves the repo is reproducible.

## Setup

```bash
python -m venv .venv
source .venv/bin/activate   # macOS/Linux
# .venv\Scripts\activate    # Windows
pip install -r requirements.txt
cp .env.example .env        # then edit .env with your values
```

## Run

```bash
python src/app.py
```
```

### 2h — Initialize Git, commit, and tag

Recall from **mini-init-git-repo**: `.gitignore` is already in place, so this is safe.

```bash
git init
git add .
git status
```

**Before committing, verify** that `git status` shows:
- ✅ `.gitignore`, `.env.example`, `README.md`, `requirements.txt`, `src/app.py` — staged
- ❌ `.env` — **not listed at all** (ignored)
- ❌ `.venv/` — **not listed at all** (ignored)

If everything looks correct:

```bash
git commit -m "Initial commit: reproducible project structure"
git tag v0.1
```

## Step 3 — Run the Checklist Programmatically

Before we simulate the clone, let's verify the project passes every checklist item. This script inspects the project directory and reports pass/fail for each check.

**Update the `PROJECT_DIR` path below** to point to the `clone-test-project` folder you just created.

In [ ]:
from pathlib import Path

# UPDATE THIS to your actual project path
PROJECT_DIR = Path.home() / "clone-test-project"  # e.g. ~/clone-test-project


def check_project(project_dir: Path) -> list[dict]:
    """Run the reproducibility checklist against a project directory."""
    results = []

    # 1. requirements.txt exists with pinned versions
    req_file = project_dir / "requirements.txt"
    if req_file.exists():
        lines = [l.strip() for l in req_file.read_text().splitlines() if l.strip() and not l.startswith("#")]
        all_pinned = all("==" in line for line in lines)
        results.append({
            "check": "requirements.txt with pinned versions",
            "pass": all_pinned,
            "detail": f"{len(lines)} packages, all pinned" if all_pinned else "Some packages missing '=='"
        })
    else:
        results.append({"check": "requirements.txt exists", "pass": False, "detail": "File not found"})

    # 2. .gitignore includes critical entries
    gi_file = project_dir / ".gitignore"
    if gi_file.exists():
        gi_content = gi_file.read_text()
        has_env = ".env" in gi_content
        has_venv = ".venv" in gi_content
        results.append({
            "check": ".gitignore covers .env and .venv/",
            "pass": has_env and has_venv,
            "detail": f".env={'✅' if has_env else '❌'}, .venv={'✅' if has_venv else '❌'}"
        })
    else:
        results.append({"check": ".gitignore exists", "pass": False, "detail": "File not found"})

    # 3. .env.example exists
    results.append({
        "check": ".env.example exists",
        "pass": (project_dir / ".env.example").exists(),
        "detail": "Tells newcomers which env vars are needed"
    })

    # 4. src/app.py exists (entry point)
    results.append({
        "check": "src/app.py entry point exists",
        "pass": (project_dir / "src" / "app.py").exists(),
        "detail": "Runnable entry point"
    })

    # 5. README.md exists
    results.append({
        "check": "README.md exists",
        "pass": (project_dir / "README.md").exists(),
        "detail": "Setup and run instructions"
    })

    # 6. .venv/ is NOT committed (check .git/index doesn't list it)
    git_dir = project_dir / ".git"
    results.append({
        "check": ".git/ directory exists (repo initialized)",
        "pass": git_dir.exists(),
        "detail": "Git repo is set up"
    })

    # 7. .env is NOT in the directory listing (should exist but be ignored)
    env_exists = (project_dir / ".env").exists()
    results.append({
        "check": ".env exists on disk (for local use)",
        "pass": env_exists,
        "detail": "Present locally but ignored by Git"
    })

    return results


results = check_project(PROJECT_DIR)

rows = "| # | Check | Result | Detail |\n|---|-------|--------|--------|\n"
all_pass = True
for i, r in enumerate(results, 1):
    status = "✅ Pass" if r["pass"] else "❌ Fail"
    if not r["pass"]:
        all_pass = False
    rows += f"| {i} | {r['check']} | {status} | {r['detail']} |\n"

verdict = "\n### ✅ All checks passed — ready for the clone test!" if all_pass else "\n### ❌ Some checks failed — fix them before proceeding."

md(f"### Reproducibility Checklist Results\n\n{rows}{verdict}")

## Step 4 — The Clone-and-Run Test

This is the moment of truth. We simulate what a stranger would do:

1. Clone the repo to a **different directory**
2. Create a **fresh** virtual environment
3. Install from `requirements.txt`
4. Copy `.env.example` to `.env`
5. Run `python src/app.py`

Since this is a local repo (not on GitHub), we'll use `git clone` with a local path. Run these commands **in a new terminal** — do NOT use the same terminal where your original project's venv is active.

### Run the clone test (terminal commands)

```bash
# 1. Clone to a separate directory (simulates a stranger's machine)
cd /tmp                              # macOS/Linux
# cd %TEMP%                          # Windows

git clone ~/clone-test-project clone-test-stranger
cd clone-test-stranger

# 2. Verify .env is NOT in the clone (secrets stayed behind)
ls -la          # .env should NOT be here

# 3. Create a fresh virtual environment
python -m venv .venv
source .venv/bin/activate            # macOS/Linux
# .venv\Scripts\activate             # Windows

# 4. Install pinned dependencies
pip install -r requirements.txt

# 5. Set up secrets from the example file
cp .env.example .env
# Optionally edit .env to change PROJECT_NAME

# 6. Run the project
python src/app.py
```

If you see the green `✅ Clone-and-run succeeded!` message, your repo is **reproducible**.

## Step 5 — Verify the Clone Programmatically

Let's also run our checklist against the **cloned** directory to confirm it has everything it needs — minus the secrets (which is correct).

In [ ]:
# Run the checklist against the cloned directory
# UPDATE this path to match where you cloned
CLONE_DIR = Path("/tmp/clone-test-stranger")  # macOS/Linux
# CLONE_DIR = Path(os.environ.get("TEMP", "")) / "clone-test-stranger"  # Windows

if CLONE_DIR.exists():
    clone_results = check_project(CLONE_DIR)

    rows = "| # | Check | Result | Detail |\n|---|-------|--------|--------|\n"
    for i, r in enumerate(clone_results, 1):
        status = "✅ Pass" if r["pass"] else "❌ Fail"
        rows += f"| {i} | {r['check']} | {status} | {r['detail']} |\n"

    # .env SHOULD be missing in the clone — that's correct behavior
    env_missing = not (CLONE_DIR / ".env").exists()
    env_note = "\n> ✅ **Correct:** `.env` is missing from the clone — secrets stayed in the original repo. " \
               "The newcomer creates their own from `.env.example`." if env_missing else \
               "\n> ❌ **Problem:** `.env` was cloned — this means it was committed. Revoke any keys in it!"

    md(f"### Clone Directory Checklist\n\n{rows}{env_note}")
else:
    print(f"Clone directory not found at {CLONE_DIR}")
    print("Run the git clone command from Step 4 first, then update CLONE_DIR.")

## Step 6 — Common Failures and How to Diagnose Them

If the clone-and-run test failed, here are the most common reasons and their fixes.

In [ ]:
failures = """
### Failure Diagnosis Guide

| Symptom | Cause | Fix |
|---------|-------|-----|
| `ModuleNotFoundError: No module named 'dotenv'` | `requirements.txt` missing or not installed | Run `pip install -r requirements.txt` in the activated venv |
| `ModuleNotFoundError: No module named 'rich'` | Forgot to `pip freeze` after installing `rich` | In original: `pip freeze > requirements.txt`, commit, re-clone |
| `FileNotFoundError: .env` | App requires `.env` but it's not in the clone | Copy `.env.example` to `.env` — this is expected behavior |
| `.env` appears in the clone | `.gitignore` was written after `.env` was committed | `git rm --cached .env`, commit, re-clone |
| `.venv/` appears in the clone | `.gitignore` was written after first commit | `git rm -r --cached .venv/`, commit, re-clone |
| `pip install` installs different versions | `requirements.txt` uses `>=` instead of `==` | Re-freeze with `pip freeze > requirements.txt` |
| `python src/app.py` says "file not found" | Wrong working directory | `cd` into the project root first |

> **Pattern:** Most failures trace back to **ordering mistakes** — doing `git add` before `.gitignore`, or `pip freeze` before installing all packages.
"""

md(failures)

## Step 7 — Clean Up

The cloned directory in `/tmp` (or `%TEMP%`) was just for testing. You can safely remove it:

```bash
rm -rf /tmp/clone-test-stranger       # macOS/Linux
# rmdir /s /q %TEMP%\clone-test-stranger   # Windows
```

Your original `clone-test-project` is your real repo — keep it as a reference template for future projects.

## 🎯 Summary

### What You Built

You built a complete, self-contained project repository and proved it is reproducible by cloning it to a separate directory, creating a fresh virtual environment, installing pinned dependencies, and running the project — all without any help from the original author.

### Key Takeaways

1. **The clone-and-run test is the ultimate reproducibility check** — if a stranger can clone your repo and run the project by following the README, your infrastructure is solid
2. **Order matters** — `.gitignore` before first commit, `pip freeze` after all packages are installed, `.env.example` committed while `.env` is ignored
3. **Secrets hygiene is verified by the clone** — if `.env` shows up in the cloned directory, it was committed and the key should be revoked immediately
4. **Dependency pinning eliminates "works on my machine"** — `pip freeze > requirements.txt` captures exact versions so the newcomer's environment is identical to yours
5. **A reproducibility checklist is a reusable tool** — run it on every project before sharing to catch missing files, unpinned dependencies, or leaked secrets